<a href="https://colab.research.google.com/github/Adonis071/AdonisPantoja/blob/main/assistente_voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install openai gtts sounddevice scipy pygame


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [2]:
pip install openai gtts sounddevice scipy pygame


In [3]:
pip install typer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.3 which is incompatible.


In [4]:
!pip install openai gtts
# Não precisamos de sounddevice ou pygame no Colab!

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [5]:
from IPython.display import HTML, Audio, display
from google.colab.output import eval_js
from base64 import b64decode
import os
from openai import OpenAI
from gtts import gTTS

# 🔑 COLOQUE SUA NOVA CHAVE AQUI (Nunca compartilhe a chave real!)
OPENAI_API_KEY = "sk-coloque-sua-nova-chave-aqui"
client = OpenAI(api_key=OPENAI_API_KEY)

# ==========================================
# 1. Gravação via JavaScript (Ouvindo o microfone pelo navegador)
# ==========================================
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_p = document.createElement("P");
var my_btn = document.createElement("BUTTON");
var t = document.createTextNode("Pressione para Falar (5 segundos)");

my_btn.appendChild(t);
my_div.appendChild(my_btn);
document.body.appendChild(my_div);

var base64data = 0;
var reader;
var recorder, gumStream;
var recordButton = my_btn;

var handleSuccess = function(stream) {
    gumStream = stream;
    var options = {
        mimeType : 'audio/webm;codecs=opus'
    };
    recorder = new MediaRecorder(stream, options);
    recorder.ondataavailable = function(e) {
        var url = URL.createObjectURL(e.data);
        var preview = document.createElement('audio');
        preview.controls = true;
        preview.src = url;
        document.body.appendChild(preview);

        reader = new FileReader();
        reader.readAsDataURL(e.data);
        reader.onloadend = function() {
            base64data = reader.result;
        }
    };
    recorder.start();
    // Para a gravação automaticamente após 5 segundos
    setTimeout(() => {
        recorder.stop();
        gumStream.getAudioTracks()[0].stop();
        recordButton.innerText = "Gravado! Aguarde o processamento...";
        recordButton.disabled = true;
    }, 5000);
};

recordButton.onclick = function() {
    recordButton.innerText = "Gravando... Fale!";
    navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);
}

function getBase64Data() {
    return new Promise((resolve) => {
        var check = setInterval(() => {
            if (base64data != 0) {
                clearInterval(check);
                resolve(base64data);
            }
        }, 500);
    });
}
</script>
"""

def gravar_audio_colab():
    print("Aparecerá um botão abaixo. Clique, permita o uso do microfone e fale por 5 segundos.")
    display(HTML(AUDIO_HTML))
    # Aguarda o JS devolver o áudio em base64
    data = eval_js("getBase64Data()")
    binary = b64decode(data.split(',')[1])

    nome_arquivo = "audio_usuario.webm"
    with open(nome_arquivo, 'wb') as f:
        f.write(binary)
    print("✅ Áudio capturado do navegador com sucesso!")
    return nome_arquivo

In [6]:
# ==========================================
# 2, 3 e 4. Processamento Padrão
# ==========================================
def processar_assistente(arquivo_audio):
    try:
        # Transcrição (Whisper)
        print("🧠 Transcrevendo...")
        with open(arquivo_audio, "rb") as f:
            transcricao = client.audio.transcriptions.create(
                model="whisper-1", file=f
            )
        texto_usuario = transcricao.text
        print(f"🗣️ Você: '{texto_usuario}'")

        # Chat (GPT)
        print("🤖 Consultando ChatGPT...")
        resposta = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Você é um assistente conciso e gentil. Responda em português de forma breve."},
                {"role": "user", "content": texto_usuario}
            ]
        )
        texto_ia = resposta.choices[0].message.content
        print(f"💬 Assistente: '{texto_ia}'")

        # Síntese de Voz (gTTS)
        print("🔊 Gerando áudio de resposta...")
        tts = gTTS(text=texto_ia, lang='pt', tld='com.br')
        arquivo_resposta = "resposta_ia.mp3"
        tts.save(arquivo_resposta)

        # Exibe o player de áudio na tela do Colab
        display(Audio(arquivo_resposta, autoplay=True))

    except Exception as e:
        print(f"❌ Erro durante o processo: {e}")

In [7]:
# ==========================================
# Execução Principal
# ==========================================
if __name__ == "__main__":
    print("🚀 Iniciando o Assistente...")
    arquivo_gravado = gravar_audio_colab()
    processar_assistente(arquivo_gravado)

🚀 Iniciando o Assistente...
Aparecerá um botão abaixo. Clique, permita o uso do microfone e fale por 5 segundos.


KeyboardInterrupt: 